# Bout and outcome parser

Convert the frozen fight_results table (one row per bout) into a clean per fighter long format table (two rows per bout, one per fighter, each carrying that fighter's individual result), apply the method taxonomy, and log every exclusion so the modelable bout count is fully accounted for.

**Input:** the frozen fight_results table (project scraper), 8,772 bouts

## Section 1: Load and inspect

Load fight_results, apply the EVENT/BOUT whitespace strip discovered in notebook 01, and print the unique OUTCOME and METHOD values before parsing. Logic assumptions will also be confirmed against the live data here.

In [ ]:
# BLOCK 1: Install and setup libraries

import pandas as pd
import numpy as np
import re
from datetime import datetime
from pathlib import Path

# data location (portable across Drive and a local repo checkout)

try:
    from google.colab import drive
    drive.mount('/content/drive')
except (ImportError, ModuleNotFoundError):
    pass  # not in Colab

DRIVE_DIR = Path('/content/drive/MyDrive/Masters in Artificial Intelligence Applied to Sport/'
                 'Masters Final Project/Pugnator mapper valorem/EDA/Code Outputs')
OUTPUT_DIR = DRIVE_DIR if DRIVE_DIR.exists() else Path('./data')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Using data directory: {OUTPUT_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# BLOCK 2: Load fight_results

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

BASE_URL = "https://raw.githubusercontent.com/th1555/ufc-fighter-value-mapper/refs/heads/main/raw_data/"

df_raw = pd.read_csv(BASE_URL + 'ufc_fight_results.csv')
print(f"Loaded fight_results: {len(df_raw):,} rows")

# Strip and clean whitespace
for col in df_raw.select_dtypes(include='object').columns:
    df_raw[col] = df_raw[col].str.strip()

print(f"\nColumns: {list(df_raw.columns)}")

Loaded fight_results: 8,772 rows

Columns: ['EVENT', 'BOUT', 'OUTCOME', 'WEIGHTCLASS', 'METHOD', 'ROUND', 'TIME', 'TIME FORMAT', 'REFEREE', 'DETAILS', 'URL']


In [ ]:
# BLOCK 3: Inspect the actual OUTCOME and METHOD values

print("Unique OUTCOME values:")
print(df_raw['OUTCOME'].value_counts(dropna=False).to_string())
print()
print("Unique METHOD values:")
print(df_raw['METHOD'].value_counts(dropna=False).to_string())

Unique OUTCOME values:
OUTCOME
W/L      5527
L/W      3090
NC/NC      90
D/D        65

Unique METHOD values:
METHOD
Decision - Unanimous       3152
KO/TKO                     2772
Submission                 1702
Decision - Split            828
Decision - Majority         104
TKO - Doctor's Stoppage      98
Overturned                   58
Could Not Continue           33
DQ                           23
Other                         2


## Section 2: Extract canonical FIGHT_ID

Each fight has a URL whose hash is the canonical fight identifier (the same extraction validated in notebook 01). This ID links the two per fighter rows of each bout so Glicko-2 can group by fight. If fight_results has no URL column, fall back to a composite key (EVENT + BOUT).

In [ ]:
# BLOCK 4: Build FIGHT_ID

# Extract canonical ID from URL column in Greco1899 scraped fight tables

url_cols = [c for c in df_raw.columns if 'url' in c.lower()]

if url_cols:
    url_col = url_cols[0]
    print(f"Using URL column '{url_col}' for FIGHT_ID extraction.")
    df_raw['FIGHT_ID'] = (
        df_raw[url_col]
        .astype(str)
        .str.rstrip('/')
        .str.split('/')
        .str[-1]
    )
else:
    print("No URL column found; building composite FIGHT_ID from EVENT + BOUT.")
    df_raw['FIGHT_ID'] = (
        df_raw['EVENT'].astype(str) + ' :: ' + df_raw['BOUT'].astype(str)
    )

# Check that FIGHT_ID is unique per bout
n_dupes = df_raw['FIGHT_ID'].duplicated().sum()
print(f"FIGHT_ID built. Duplicates: {n_dupes}")
if n_dupes > 0:
    print("WARN: duplicate FIGHT_IDs exist; inspect before relying on the key.")
    print(df_raw[df_raw['FIGHT_ID'].duplicated(keep=False)][['EVENT', 'BOUT', 'FIGHT_ID']].head())

Using URL column 'URL' for FIGHT_ID extraction.
FIGHT_ID built. Duplicates: 0


## Section 3: Classify methods and outcomes

Define the taxonomy that determines which bouts are modelable. Each METHOD maps to a category (decision / finish / dq / excluded). Each OUTCOME maps to a result pair. Bouts that are No Contest, Overturned, Could Not Continue, or Other are excluded from the modelling base and logged.

In [ ]:
# BLOCK 5: Taxonomy lookups

# Map the raw string values to categories

# METHOD categories
METHOD_CATEGORIES = [
    ('decision - unanimous', 'decision', True),
    ('decision - split',     'decision', True),
    ('decision - majority',  'decision', True),
    ('decision',             'decision', True),
    ('ko/tko',               'finish',   True),
    ('tko',                  'finish',   True),
    ('ko',                   'finish',   True),
    ('submission',           'finish',   True),
    ("doctor's stoppage",    'finish',   True),
    ('dq',                   'dq',       True),
    ('disqualification',     'dq',       True),
    ('overturned',           'excluded', False),
    ('could not continue',   'excluded', False),
    ('other',                'excluded', False),
]

def classify_method(method_str):
    if pd.isna(method_str):
        return ('missing', False)
    m = str(method_str).lower()
    for substring, category, modelable in METHOD_CATEGORIES:
        if substring in m:
            return (category, modelable)
    return ('unmatched', False)

# Test against the actual method values
print("Method classification preview:")
method_preview = df_raw['METHOD'].dropna().unique()
for m in sorted(method_preview):
    cat, modelable = classify_method(m)
    flag = 'modelable' if modelable else 'EXCLUDED'
    print(f"  {str(m)[:40]:40s} -> {cat:10s} ({flag})")

Method classification preview:
  Could Not Continue                       -> excluded   (EXCLUDED)
  DQ                                       -> dq         (modelable)
  Decision - Majority                      -> decision   (modelable)
  Decision - Split                         -> decision   (modelable)
  Decision - Unanimous                     -> decision   (modelable)
  KO/TKO                                   -> finish     (modelable)
  Other                                    -> excluded   (EXCLUDED)
  Overturned                               -> excluded   (EXCLUDED)
  Submission                               -> finish     (modelable)
  TKO - Doctor's Stoppage                  -> finish     (modelable)


In [ ]:
# BLOCK 6: Outcome parsing

# OUTCOME is positional and split by '/'
#   'W/L' > first fighter won, second lost
#   'L/W' > first lost, second won
#   'D/D' > draw
#   'NC/NC' > no contest (excluded)

def parse_outcome(outcome_str):
    if pd.isna(outcome_str):
        return (None, None, False)
    parts = str(outcome_str).strip().split('/')
    if len(parts) != 2:
        return (None, None, False)
    a, b = parts[0].strip().upper(), parts[1].strip().upper()
    if 'NC' in (a, b):
        return (a, b, False)
    # Valid result tokens
    valid = {'W', 'L', 'D'}
    if a in valid and b in valid:
        return (a, b, True)
    return (a, b, False)  # unexpected token

# Preview against actual outcome values
print("Outcome parsing preview:")
for o in sorted(df_raw['OUTCOME'].dropna().unique()):
    ra, rb, ok = parse_outcome(o)
    flag = 'modelable' if ok else 'EXCLUDED'
    print(f"  {str(o):10s} -> ({ra}, {rb}) ({flag})")

Outcome parsing preview:
  D/D        -> (D, D) (modelable)
  L/W        -> (L, W) (modelable)
  NC/NC      -> (NC, NC) (EXCLUDED)
  W/L        -> (W, L) (modelable)


## Section 4: Parse to fighter long format

For each bout: split the BOUT into two fighters, parse the OUTCOME, classify the METHOD, and decide whether the bout is modelable. Excluded bouts produce a log entry with the reason. Nothing is dropped silently.

In [ ]:
# BLOCK 7: Main parser

# Iterate to give usable fighter rows (modelable) or a
# exclusion log entry (not modelable)

def map_result_to_score(result):
    return {'W': 1.0, 'L': 0.0, 'D': 0.5}.get(result, None)

per_fighter_rows = []
exclusion_log = []

for _, fight in df_raw.iterrows():
    fight_id = fight['FIGHT_ID']
    bout = fight['BOUT']
    outcome = fight['OUTCOME']
    method = fight['METHOD']
    event = fight['EVENT']
    weightclass = fight.get('WEIGHTCLASS', None)

    # Parse the BOUT into two fighters
    if pd.isna(bout) or ' vs. ' not in str(bout):
        exclusion_log.append({'FIGHT_ID': fight_id, 'BOUT': bout,
                              'reason': 'malformed_bout', 'method': method,
                              'outcome': outcome})
        continue
    bout_parts = str(bout).split(' vs. ')
    if len(bout_parts) != 2:
        exclusion_log.append({'FIGHT_ID': fight_id, 'BOUT': bout,
                              'reason': 'malformed_bout', 'method': method,
                              'outcome': outcome})
        continue
    fighter_a, fighter_b = bout_parts[0].strip(), bout_parts[1].strip()

    # Classify the method
    method_cat, method_modelable = classify_method(method)
    result_a, result_b, outcome_ok = parse_outcome(outcome)

    # Collect ALL applicable exclusion reasons
    reasons = []
    if not method_modelable:
        reasons.append(f'method_{method_cat}')
    if not outcome_ok:
        if result_a == 'NC' or result_b == 'NC':
            reasons.append('outcome_nc')
        else:
            reasons.append('outcome_not_modelable')

    # log the combined reason
    if reasons:
        exclusion_log.append({
            'FIGHT_ID': fight_id, 'BOUT': bout,
            'reason': ';'.join(reasons),
            'method': method, 'outcome': outcome,
        })
        continue

    # Modelable
    is_title = isinstance(weightclass, str) and 'title' in weightclass.lower()
    is_dq = (method_cat == 'dq')

    per_fighter_rows.append({
        'FIGHT_ID': fight_id, 'FIGHTER': fighter_a, 'OPPONENT': fighter_b,
        'RESULT': result_a, 'SCORE': map_result_to_score(result_a),
        'METHOD': method, 'METHOD_CAT': method_cat, 'IS_DQ': is_dq,
        'EVENT': event, 'WEIGHTCLASS': weightclass, 'IS_TITLE': is_title,
    })
    per_fighter_rows.append({
        'FIGHT_ID': fight_id, 'FIGHTER': fighter_b, 'OPPONENT': fighter_a,
        'RESULT': result_b, 'SCORE': map_result_to_score(result_b),
        'METHOD': method, 'METHOD_CAT': method_cat, 'IS_DQ': is_dq,
        'EVENT': event, 'WEIGHTCLASS': weightclass, 'IS_TITLE': is_title,
    })

df_per_fighter = pd.DataFrame(per_fighter_rows)
df_exclusions = pd.DataFrame(exclusion_log)

print(f"Fighter rows produced: {len(df_per_fighter):,}")
print(f"Modelable bouts: {len(df_per_fighter)//2:,}")
print(f"Excluded bouts: {len(df_exclusions):,}")

Fighter rows produced: 17,358
Modelable bouts: 8,679
Excluded bouts: 93


## Section 5: Reconciliation and validation

Confirm that modelable bouts plus excluded bouts equal the original row count (nothing dropped silently). Break down the exclusions by reason.

In [ ]:
# BLOCK 8: Reconciliation

n_original = len(df_raw)
n_modelable = len(df_per_fighter) // 2
n_excluded = len(df_exclusions)

print("RECONCILIATION")
print(f"  Original bouts:   {n_original:,}")
print(f"  Modelable bouts:  {n_modelable:,}")
print(f"  Excluded bouts:   {n_excluded:,}")
print(f"  Sum check:        {n_modelable + n_excluded:,} (should equal {n_original:,})")

if n_modelable + n_excluded == n_original:
    print("  OK: every input bout is accounted for.")
else:
    print("  WARN: counts do not reconcile; investigate.")

print()
print("Exclusions by reason:")
if len(df_exclusions) > 0:
    print(df_exclusions['reason'].value_counts().to_string())

print()
print(f"Expected modelable (from project memory): ~8,580")
print(f"Actual modelable: {n_modelable:,}")
diff = abs(n_modelable - 8580)
if diff <= 100:
    print(f"  OK: within 100 of expected (diff {diff}). Data has grown since the memory note was written.")
else:
    print(f"  CHECK: diff of {diff} from expected; review the exclusion breakdown above.")

RECONCILIATION
  Original bouts:   8,772
  Modelable bouts:  8,679
  Excluded bouts:   93
  Sum check:        8,772 (should equal 8,772)
  OK: every input bout is accounted for.

Exclusions by reason:
reason
method_excluded;outcome_nc    90
method_excluded                3

Expected modelable (from project memory): ~8,580
Actual modelable: 8,679
  OK: within 100 of expected (diff 99). Data has grown since the memory note was written.


In [ ]:
# BLOCK 9: Check the fighter output

print("Per-fighter dataframe sample:")
print(df_per_fighter.head(6).to_string())
print()

# Each FIGHT_ID should appear twice
fight_counts = df_per_fighter['FIGHT_ID'].value_counts()
not_two = fight_counts[fight_counts != 2]
print(f"FIGHT_IDs not appearing exactly twice: {len(not_two)}")
if len(not_two) > 0:
    print("  WARN: some bouts do not have exactly two fighter rows.")
    print(not_two.head())

# Scores should sum to 1.0 (W+L) or 1.0 (D+D = 0.5+0.5)
score_sums = df_per_fighter.groupby('FIGHT_ID')['SCORE'].sum()
bad_sums = score_sums[(score_sums - 1.0).abs() > 0.001]
print(f"\nBouts where fighter scores don't sum to 1.0: {len(bad_sums)}")
if len(bad_sums) > 0:
    print("  WARN: score asymmetry; inspect.")
    print(bad_sums.head())
else:
    print("  OK: all bouts have complementary scores.")

print()
print("Result distribution:")
print(df_per_fighter['RESULT'].value_counts().to_string())
print()
print("Method category distribution:")
print(df_per_fighter['METHOD_CAT'].value_counts().to_string())

Per-fighter dataframe sample:
           FIGHT_ID            FIGHTER           OPPONENT RESULT  SCORE                METHOD METHOD_CAT  IS_DQ                             EVENT         WEIGHTCLASS  IS_TITLE
0  e4aa608124896794       Arnold Allen   Melquizael Costa      W    1.0  Decision - Unanimous   decision  False  UFC Fight Night: Allen vs. Costa  Featherweight Bout     False
1  e4aa608124896794   Melquizael Costa       Arnold Allen      L    0.0  Decision - Unanimous   decision  False  UFC Fight Night: Allen vs. Costa  Featherweight Bout     False
2  fc1266e2892ed111         Dooho Choi      Daniel Santos      W    1.0                KO/TKO     finish  False  UFC Fight Night: Allen vs. Costa  Featherweight Bout     False
3  fc1266e2892ed111      Daniel Santos         Dooho Choi      L    0.0                KO/TKO     finish  False  UFC Fight Night: Allen vs. Costa  Featherweight Bout     False
4  ecb7ff543dd41bf8  Malcolm Wellmaker          Juan Diaz      L    0.0            Submiss

In [ ]:
# BLOCK 10: Weight class analysis and filter flags

# The modelable base from the parser above includes all weight classes.
# Open Weight and Catch Weight are excluded for Spearman validation but included
# for overall Glicko-2; adds a flag column so downstream notebooks can filter

print("WEIGHT CLASS BREAKDOWN (modelable bouts)")
print()
wc_counts = df_per_fighter['WEIGHTCLASS'].value_counts(dropna=False)
print(wc_counts.to_string())

# Flag Open Weight and Catch Weight
df_per_fighter['DIVISION_VALID'] = ~df_per_fighter['WEIGHTCLASS'].str.contains(
    'open weight|catch weight', case=False, na=False
)

n_valid = df_per_fighter['DIVISION_VALID'].sum()
n_excluded = (~df_per_fighter['DIVISION_VALID']).sum()
print(f"\nDivision valid: {n_valid:,} rows ({n_valid//2:,} bouts)")
print(f"Open/Catch Weight flagged: {n_excluded:,} rows ({n_excluded//2:,} bouts)")

# Title fight count
n_title = df_per_fighter['IS_TITLE'].sum()
print(f"\nTitle bout rows: {n_title:,} ({n_title//2:,} bouts)")

WEIGHT CLASS BREAKDOWN (modelable bouts)

WEIGHTCLASS
Lightweight Bout                                                        2790
Welterweight Bout                                                       2652
Middleweight Bout                                                       2176
Featherweight Bout                                                      1660
Bantamweight Bout                                                       1484
Heavyweight Bout                                                        1408
Light Heavyweight Bout                                                  1374
Flyweight Bout                                                           778
Women's Strawweight Bout                                                 698
Women's Flyweight Bout                                                   522
Women's Bantamweight Bout                                                452
Open Weight Bout                                                         202
Catch Weight Bout     

In [ ]:
# BLOCK 11: Era partition

# Label each fight with a broadcast era for normalisation on the public profile axis:
#   pre_2026  = PPV era (Fox 2012-2018 + ESPN 2019-2025; both PPV-based)
#   post_2026 = post-PPV era (Paramount+ subscription, no US PPV)

# Isolate Dates
df_events = pd.read_csv(BASE_URL + 'ufc_event_details.csv')
for col in df_events.select_dtypes(include='object').columns:
    df_events[col] = df_events[col].str.strip()
df_events['DATE_PARSED'] = pd.to_datetime(df_events['DATE'], errors='coerce')

# Join dates onto the fighter frame via EVENT
df_per_fighter = df_per_fighter.merge(
    df_events[['EVENT', 'DATE_PARSED']],
    on='EVENT',
    how='left'
)

# Check join success
n_with_date = df_per_fighter['DATE_PARSED'].notna().sum()
print(f"Date attached: {n_with_date:,}/{len(df_per_fighter):,} rows ({100*n_with_date/len(df_per_fighter):.1f}%)")

# Apply the era partition
ERA_CUTOFF = pd.Timestamp('2026-01-01')

df_per_fighter['ERA'] = np.where(
    df_per_fighter['DATE_PARSED'] < ERA_CUTOFF,
    'pre_2026',
    'post_2026'
)
# Rows with no date get 'unknown'
df_per_fighter.loc[df_per_fighter['DATE_PARSED'].isna(), 'ERA'] = 'unknown'

print()
print("Era distribution:")
era_counts = df_per_fighter['ERA'].value_counts()
print(era_counts.to_string())
print()
print(f"Pre-2026 bouts:  {era_counts.get('pre_2026', 0)//2:,}")
print(f"Post-2026 bouts: {era_counts.get('post_2026', 0)//2:,}")
if 'unknown' in era_counts.index:
    print(f"Unknown (no date): {era_counts.get('unknown', 0)//2:,}")

Date attached: 17,358/17,358 rows (100.0%)

Era distribution:
ERA
pre_2026     16808
post_2026      550

Pre-2026 bouts:  8,404
Post-2026 bouts: 275


In [ ]:
# BLOCK 12: Canonical weight class derivation

# The 12 real UFC weight classes

_REAL_DIVISIONS = [
    "Women's Strawweight", "Women's Flyweight", "Women's Bantamweight",
    "Women's Featherweight",
    "Light Heavyweight", "Heavyweight", "Middleweight", "Welterweight",
    "Lightweight", "Featherweight", "Bantamweight", "Flyweight", "Strawweight",
]

def canonicalise_weightclass(wc):
    if pd.isna(wc):
        return None
    text = str(wc)
    # handle tournament and season labels (TUF, Road to UFC, Ultimate Japan, numbered
    # tournaments) which carry the real division inside a longer string
    if re.search(r'\b(Tournament|Ultimate Fighter|Ultimate Japan|Road to)\b', text, re.IGNORECASE):
        for div in _REAL_DIVISIONS:
            if div.lower() in text.lower():
                return div
        # drop a tournament string with no recognisable division
        return None
    # strip the qualifier words
    cleaned = re.sub(r'\b(UFC|Interim|Title|Bout)\b', '', text, flags=re.IGNORECASE)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned if cleaned else None

df_per_fighter['WEIGHTCLASS_CANONICAL'] = df_per_fighter['WEIGHTCLASS'].apply(canonicalise_weightclass)

print("Canonical weight class distribution:")
print(df_per_fighter['WEIGHTCLASS_CANONICAL'].value_counts(dropna=False).to_string())
print()
print(f"Distinct values: {df_per_fighter['WEIGHTCLASS_CANONICAL'].nunique()}")

Canonical weight class distribution:
WEIGHTCLASS_CANONICAL
Lightweight                2898
Welterweight               2784
Middleweight               2286
Featherweight              1726
Bantamweight               1554
Heavyweight                1532
Light Heavyweight          1500
Flyweight                   840
Women's Strawweight         742
Women's Flyweight           550
Women's Bantamweight        494
Open Weight                 202
Catch Weight                162
Women's Featherweight        60
None                         20
Superfight Championship       6
Super Heavyweight             2

Distinct values: 16


## Section 6: Save outputs

Create table and the exclusion log. Downstream conversion and modelling notebooks read these.

In [ ]:
# BLOCK 13: Save outputs

# /content/drive/MyDrive/Masters in Artificial Intelligence Applied to Sport/Masters Final Project/Pugnator mapper valorem/EDA/Code Outputs
OUTPUT_DIR = Path('/content/drive/MyDrive/Masters in Artificial Intelligence Applied to Sport/Masters Final Project/Pugnator mapper valorem/EDA/Code Outputs')

# Create the output directory if it doesn't exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

per_fighter_path = OUTPUT_DIR / 'fights_per_fighter.parquet'
exclusion_path = OUTPUT_DIR / 'bout_exclusion_log.csv'

df_per_fighter.to_parquet(per_fighter_path, index=False)
df_exclusions.to_csv(exclusion_path, index=False)

print(f"Saved: {per_fighter_path}")
print(f"  ({len(df_per_fighter):,} rows = {len(df_per_fighter)//2:,} modelable bouts x 2)")
print(f"Saved: {exclusion_path}")
print(f"  ({len(df_exclusions):,} excluded bouts)")

Saved: /content/drive/MyDrive/Masters in Artificial Intelligence Applied to Sport/Masters Final Project/Pugnator mapper valorem/EDA/Code Outputs/fights_per_fighter.parquet
  (17,358 rows = 8,679 modelable bouts x 2)
Saved: /content/drive/MyDrive/Masters in Artificial Intelligence Applied to Sport/Masters Final Project/Pugnator mapper valorem/EDA/Code Outputs/bout_exclusion_log.csv
  (93 excluded bouts)


In [ ]:
print('WEIGHTCLASS_CANONICAL' in df_per_fighter.columns)
print(df_per_fighter['WEIGHTCLASS_CANONICAL'].value_counts().to_string())

True
WEIGHTCLASS_CANONICAL
Lightweight                2898
Welterweight               2784
Middleweight               2286
Featherweight              1726
Bantamweight               1554
Heavyweight                1532
Light Heavyweight          1500
Flyweight                   840
Women's Strawweight         742
Women's Flyweight           550
Women's Bantamweight        494
Open Weight                 202
Catch Weight                162
Women's Featherweight        60
Superfight Championship       6
Super Heavyweight             2


## Section 7: Summary

The parser converts 8,772 original bouts into 17,358 per fighter rows (8,679 modelable bouts, each producing two rows). Every input bout is accounted for: 8,679 modelable plus 93 excluded equals the 8,772 total. The 93 exclusions are logged with reasons: 90 method-excluded or No Contest, and 3 method-excluded only.

**Weight class flags:** 16,994 rows (8,497 bouts) carry a valid canonical division; 364 rows (182 bouts) are flagged Open or Catch Weight; 948 rows (474 bouts) are title bouts.

**Era partition:** every row carries a bout date; 16,808 rows fall in the pre-2026 (Fox and ESPN) era and 550 in the Paramount+ era from January 2026.

**Outputs:** fights_per_fighter.parquet, bout_exclusion_log.csv